# Local Evaluation on RTX 4050

Load the LoRA adapter downloaded from Kaggle, run inference on `Validation.csv` (or any
held-out CSV with the same schema), report per-subset ROUGE.

**Before running:**

1. Place the downloaded `lora_adapter/` directory somewhere on your laptop.
   It should contain `adapter_model.safetensors` (or `.bin`), `adapter_config.json`,
   tokenizer files (`spiece.model`, `tokenizer_config.json`, etc.), and `length_stats.json`.
2. Place `Validation.csv` accessible on disk.
3. Update paths in the **Config** cell below.

**Expected runtime on RTX 4050 (6GB) with beam=4 over the full validation set: ~10–20 minutes.**

This notebook does:
- Load base mT0-large + your LoRA adapter
- Read per-subset length stats from the adapter directory
- Generate predictions with per-subset `max_new_tokens`
- Compute and print per-subset ROUGE
- Save `val_predictions.csv` and `val_metrics.json` locally


In [ ]:
# Install in your venv before running this notebook. Sample line:
# pip install transformers peft datasets rouge_score sentencepiece pandas numpy torch tqdm
# (If torch is missing, install matching your CUDA version from https://pytorch.org)

In [2]:
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"CUDA: {torch.cuda.is_available()} | device: {device_name}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu128).
W0528 03:57:19.752000 16524 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


CUDA: True | device: NVIDIA GeForce RTX 4050 Laptop GPU


## Config — edit paths to match your machine

In [3]:
# ---- Paths (EDIT THESE) ----
BASE_MODEL = "./mt0-large"
ADAPTER_DIR = Path("./lora_adapter")            # downloaded from Kaggle
VAL_CSV = Path("./Val.csv")
OUTPUT_DIR = Path("./local_eval_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---- Inference settings — should match training ----
SUBSET_PREFIX_FORMAT = "[{subset}] "
MAX_INPUT_LENGTH = 256
DEFAULT_MAX_NEW_TOKENS = 512
NUM_BEAMS = 1
LENGTH_PENALTY = 1.0
NO_REPEAT_NGRAM = 3
BATCH_SIZE = 16                              # bump if VRAM allows; nudge down on OOM

# Load per-subset length stats saved alongside the adapter
with open(ADAPTER_DIR / "length_stats.json") as f:
    length_stats = json.load(f)
per_subset_max_length = {s: int(v["p95"]) + 32 for s, v in length_stats.items()}

print("Per-subset max_new_tokens:")
for s, m in sorted(per_subset_max_length.items()):
    print(f"  {s}: {m}")

Per-subset max_new_tokens:
  Aka_Gha: 531
  Amh_Eth: 148
  Eng_Eth: 96
  Eng_Gha: 264
  Eng_Ken: 301
  Eng_Uga: 415
  Lug_Uga: 534
  Swa_Ken: 410


## Load tokenizer, base model, and LoRA adapter

In [4]:
# Prefer the tokenizer saved alongside the adapter (in case anything was changed)
try:
    tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), use_fast=True)
    print(f"Loaded tokenizer from adapter dir")
except Exception as e:
    print(f"Could not load tokenizer from adapter ({e}); falling back to {BASE_MODEL}")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

# RTX 4050 supports bf16 natively (Ada Lovelace).
base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB total")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loaded tokenizer from adapter dir


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


GPU memory: 2.52 GB allocated / 6.44 GB total


## Load validation data

In [5]:
val_df = pd.read_csv(VAL_CSV).dropna(subset=["input", "output", "subset"])
val_df = val_df[val_df["input"].astype(str).str.strip().str.len() > 0]
val_df = val_df[val_df["output"].astype(str).str.strip().str.len() > 0].reset_index(drop=True)
val_df["input_with_prefix"] = val_df.apply(
    lambda r: SUBSET_PREFIX_FORMAT.format(subset=r["subset"]) + str(r["input"]),
    axis=1,
)
print(f"Validation: {len(val_df)} rows")
print(val_df["subset"].value_counts())

Validation: 6686 rows
subset
Eng_Uga    1688
Aka_Gha    1114
Eng_Gha    1104
Lug_Uga     846
Eng_Eth     564
Swa_Ken     518
Amh_Eth     462
Eng_Ken     390
Name: count, dtype: int64


In [6]:
print("device:", next(model.parameters()).device)   # want cuda:0

print("dtype:", next(model.parameters()).dtype)      # want torch.bfloat16

device: cuda:0
dtype: torch.bfloat16


## Generate predictions (per-subset `max_new_tokens`)

In [7]:
@torch.no_grad()
def generate_predictions(model, tokenizer, df, per_subset_max, batch_size=4):
    device = next(model.parameters()).device
    out_rows = []
    for subset, group in df.groupby("subset"):
        max_new = per_subset_max.get(subset, DEFAULT_MAX_NEW_TOKENS)
        rows = group.to_dict("records")
        for i in tqdm(range(0, len(rows), batch_size), desc=f"gen[{subset}] (max={max_new})"):
            batch = rows[i:i+batch_size]
            inputs = tokenizer(
                [r["input_with_prefix"] for r in batch],
                max_length=MAX_INPUT_LENGTH,
                truncation=True, padding=True, return_tensors="pt",
            ).to(device)
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new,
                num_beams=NUM_BEAMS,
                length_penalty=LENGTH_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM,
                early_stopping=True,
            )
            decoded = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
            for r, p in zip(batch, decoded):
                out_rows.append({
                    "ID": r.get("ID"),
                    "subset": r["subset"],
                    "input": r["input"],
                    "reference": r["output"],
                    "prediction": p.strip(),
                })
    return pd.DataFrame(out_rows)

pred_df = generate_predictions(model, tokenizer, val_df, per_subset_max_length, batch_size=BATCH_SIZE)
pred_df.to_csv(OUTPUT_DIR / "val_predictions.csv", index=False)
print(f"\nSaved {len(pred_df)} predictions to {OUTPUT_DIR / 'val_predictions.csv'}")

gen[Aka_Gha] (max=531):   0%|          | 0/70 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


gen[Amh_Eth] (max=148):   0%|          | 0/29 [00:00<?, ?it/s]

gen[Eng_Eth] (max=96):   0%|          | 0/36 [00:00<?, ?it/s]

gen[Eng_Gha] (max=264):   0%|          | 0/69 [00:00<?, ?it/s]

gen[Eng_Ken] (max=301):   0%|          | 0/25 [00:00<?, ?it/s]

gen[Eng_Uga] (max=415):   0%|          | 0/106 [00:00<?, ?it/s]

gen[Lug_Uga] (max=534):   0%|          | 0/53 [00:00<?, ?it/s]

gen[Swa_Ken] (max=410):   0%|          | 0/33 [00:00<?, ?it/s]


Saved 6686 predictions to local_eval_outputs\val_predictions.csv


## Per-subset ROUGE

In [8]:
METRIC_KEYS = ("rouge1", "rouge2", "rougeL", "rougeLsum")

def compute_per_subset_rouge(preds, refs, subs, use_stemmer=False):
    scorer = rouge_scorer.RougeScorer(list(METRIC_KEYS), use_stemmer=use_stemmer)
    per_sample = {k: [] for k in METRIC_KEYS}
    for p, r in zip(preds, refs):
        scores = scorer.score(r, p)
        for k in METRIC_KEYS:
            per_sample[k].append(scores[k].fmeasure)
    per_subset = defaultdict(lambda: defaultdict(list))
    for i, s in enumerate(subs):
        for k in METRIC_KEYS:
            per_subset[s][k].append(per_sample[k][i])
    summary = {s: {k: float(np.mean(v)) for k, v in m.items()} for s, m in per_subset.items()}
    for s in summary:
        summary[s]["count"] = len(per_subset[s]["rouge1"])
    overall_micro = {k: float(np.mean(per_sample[k])) for k in METRIC_KEYS}
    overall_macro = {k: float(np.mean([summary[s][k] for s in summary])) for k in METRIC_KEYS}
    return {"overall_micro": overall_micro, "overall_macro": overall_macro, "per_subset": summary}

results = compute_per_subset_rouge(
    pred_df["prediction"].tolist(),
    pred_df["reference"].tolist(),
    pred_df["subset"].tolist(),
)
with open(OUTPUT_DIR / "val_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("=" * 80)
print("OVERALL (micro):")
for k in METRIC_KEYS:
    print(f"  {k:<12} {results['overall_micro'][k]:.4f}")
print("\nOVERALL (macro across subsets):")
for k in METRIC_KEYS:
    print(f"  {k:<12} {results['overall_macro'][k]:.4f}")
print("\nPER-SUBSET:")
print(f"{'Subset':<12} {'N':>6} {'ROUGE-1':>10} {'ROUGE-2':>10} {'ROUGE-L':>10} {'ROUGE-Lsum':>12}")
print("-" * 80)
for s in sorted(results["per_subset"].keys()):
    m = results["per_subset"][s]
    print(f"{s:<12} {m['count']:>6d} {m['rouge1']:>10.4f} {m['rouge2']:>10.4f} {m['rougeL']:>10.4f} {m['rougeLsum']:>12.4f}")
print("=" * 80)

OVERALL (micro):
  rouge1       0.2324
  rouge2       0.0799
  rougeL       0.1681
  rougeLsum    0.1681

OVERALL (macro across subsets):
  rouge1       0.2186
  rouge2       0.0753
  rougeL       0.1614
  rougeLsum    0.1614

PER-SUBSET:
Subset            N    ROUGE-1    ROUGE-2    ROUGE-L   ROUGE-Lsum
--------------------------------------------------------------------------------
Aka_Gha        1114     0.3763     0.1454     0.2447       0.2447
Amh_Eth         462     0.0228     0.0005     0.0228       0.0228
Eng_Eth         564     0.2951     0.1418     0.2504       0.2504
Eng_Gha        1104     0.3183     0.1375     0.2367       0.2367
Eng_Ken         390     0.1912     0.0470     0.1409       0.1409
Eng_Uga        1688     0.1721     0.0370     0.1220       0.1220
Lug_Uga         846     0.1478     0.0340     0.1087       0.1087
Swa_Ken         518     0.2249     0.0594     0.1652       0.1652


## Eyeball a few generations per subset

In [11]:
for s in sorted(pred_df["subset"].unique()):
    row = pred_df[pred_df["subset"] == s].iloc[1]
    print(f"\n--- {s} ---")
    print(f"Q:    {row['input'][:200]}")
    print(f"REF:  {row['reference'][:250]}")
    print(f"PRED: {row['prediction'][:250]}")


--- Aka_Gha ---
Q:    Dɛn nti na ɛho hia sɛ mmabun te wɔn nna ne awo hokwan ahorow ase?
REF:  Nna ne awo hokwan ahorow a wɔte ase no ma mmabun tumi: Si gyinae a ɛfata wɔ wɔn nipadua, nna, ne abusuabɔ ho. Kamfo wɔn hokwan ahorow ne nna ne awo akwahosan ho nhyehyɛe ne nsɛm a ɛho hia a wobenya. Bɔ wɔn ho ban fi nhyɛso, nyiyim, asisifo ne basabas
PRED: Mmabun a wɔde wɔn nna ne awo hokwan ahorow ase a ɛho hia sɛ wɔte wɔn ho nsɛm a wobɛma wɔn ahorow a wonnim. Ɛho yɛ hia ne sɛnea wɔbɛma nnipa ayɛ nna ho hokwaan ahorow, a wode wɔn wɔn so a, ɛka sɔ sԑ wɔbɛte wɔ nna a woreyɛ nkɔmmɔbɔ a.

--- Amh_Eth ---
Q:    በተደጋጋሚ በሚከሰት የሄርፒስ ኢንፌክሽን ምክንያት የተለመደው የቁስል ገጽታ ምን ይመስላል?
REF:  ብዙውን ጊዜ በመጠን እና በቁጥር ያነሱ ናቸው፤ በፍጥነት ይድናሉ፤ እናም ከመጀመሪያውው ኢንፌክሽን ይልቅ በትንሽ ቦታ የተወሰኑ ናቸው።
PRED: ቁስል ገጽታ ሁሉም ሰውነት ላይ ቆስል ያደርገዋል።

--- Eng_Eth ---
Q:    How do people get Does trichomoniasis always cause signs?
REF:  This is a question about, Trichomoniasis. No. Some people have no signs, but others notice itching, bad smell, or pain

In [ ]:
import re
import numpy as np
from rouge_score import rouge_scorer
from rouge_score.tokenizers import Tokenizer

class MultilingualTokenizer(Tokenizer):
    def tokenize(self, text):
        text = text.lower()
        text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
        return [t for t in text.split() if t]

scorer = rouge_scorer.RougeScorer(["rouge1"], tokenizer=MultilingualTokenizer())

scores = [
    scorer.score(ref, pred)["rouge1"].fmeasure
    for ref, pred in zip(pred_df["reference"], pred_df["prediction"])
]
print(f"Overall ROUGE-1 (micro, multilingual): {np.mean(scores):.4f}")

Overall ROUGE-1 (micro, multilingual): 0.2267


: 

## What to look for in the results

- **Aka_Gha and Lug_Uga ROUGE-L vs the English subsets**: this is the gap that tells you
  whether to invest in AfriTeVa V2 + DAPT on those languages. A 5+ point gap is a clear
  signal; a 1–2 point gap might just be irreducible.
- **ROUGE-1 way higher than ROUGE-L**: indicates token-level overlap without much
  structural match — model is hitting the right words but not in the right order. Try a
  lower length_penalty (0.6–0.8) to encourage shorter, more structured outputs.
- **Predictions that look like the input copied back**: trained too few epochs or the
  LoRA rank is too low. Bump `LORA_R` to 32 and/or epochs to 4–5 and retrain.
- **Predictions in English when the input is in a non-English language** (or vice versa):
  the subset prefix isn't being respected. Sanity-check that the prefix was correctly
  applied during training and is present here.

Save the entire `local_eval_outputs/` directory and commit it to your experiment log so
you can compare against the next run.